# Unit 9 — Diffusion policy training: noise, objective, update

**Your result:** the functions that train a diffusion policy: cumulative alphas, forward noising, timestep selection, a masked noise loss, and a loss callback that the training loop uses to update a real denoiser. For the same random numbers your loss equals the pinned LeRobot loss.

Five short exercises (10–20 minutes each) and a larger experiment (30–60 minutes). These times are estimates, not measurements. Full answers are in [the reference notebook](09-diffusion-training-reference.ipynb).

In [ ]:
# Supplied: workspace, offline caches, imports
from pathlib import Path
import os
import sys


def find_root():
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (path / "manifests" / "act_references.yaml").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the repository tree.")


WORKSPACE = find_root()
if str(WORKSPACE / "src") not in sys.path:
    sys.path.insert(0, str(WORKSPACE / "src"))

# Set cache paths and offline mode before any data or model library is imported.
os.environ["HF_HUB_CACHE"] = str(WORKSPACE / ".local-data/hf/hub")
os.environ["HF_DATASETS_CACHE"] = str(WORKSPACE / ".local-data/hf/datasets")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ.setdefault("TORCH_HOME", str(WORKSPACE / ".local-data/torch"))
CONFIGS, DATA = WORKSPACE / "configs", WORKSPACE / ".local-data"
import math
import time
import torch
from torch import nn
import matplotlib.pyplot as plt

import vla_lab.teaching.diffusion as ref  # package reference: used by checks only
from vla_lab.teaching.diffusion import (
    ConditionalUnet1D, DDPMSchedule, DiffusionDenoiser, LeRobotDenoiser, cosine_betas,
    flat_state_condition, load_state_action_windows, stack_observation, take_batch,
)
from vla_lab.teaching.training import optimizer_update, train_loop  # from the training unit (package versions)
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
from lerobot.configs import FeatureType, PolicyFeature
from lerobot.policies.diffusion import DiffusionConfig, DiffusionPolicy

torch.manual_seed(0)
print("PyTorch:", torch.__version__)
print("Workspace:", WORKSPACE)

In [ ]:
# Supplied: a helper that the checks use
def close(actual, expected, message, atol=1e-5):
    """Compare two tensors and say what failed."""
    assert actual.shape == expected.shape, f"{message}: shape {tuple(actual.shape)}, expected {tuple(expected.shape)}"
    worst = (actual.detach() - expected.detach()).abs().max().item()
    assert worst <= atol, f"{message}: largest difference {worst:.2e} is above {atol:.0e}"

## Background: the forward process and what the network learns

Start from a clean action window `x0` (16 steps × 7 values, scaled to `[-1, 1]`). The **forward process** adds Gaussian noise in `T = 100` small steps. A closed form jumps straight to step `t`:

`x_t = sqrt(ᾱ_t) · x0 + sqrt(1 − ᾱ_t) · ε`, with `ε ~ N(0, I)` and `ᾱ_t = (1 − β_0)(1 − β_1)…(1 − β_t)`.

Small `t` is almost clean; large `t` is almost pure noise. The **network** `ε_θ(x_t, t, observation)` is trained to predict the noise `ε` that was added. The loss is the mean squared error over *real* action steps. Sampling later runs the process in reverse (Unit 11).

The pinned recipe, from `lerobot/policies/diffusion` at revision `e0d50211ef236143ae867228662b7dfaba554f02`: `T = 100`, the `squaredcos_cap_v2` (cosine) beta schedule, noise prediction, padded steps masked (`do_mask_loss_for_padding=True`), observation history 2, horizon 16, 8 executed actions. The current trainer uses Adam with a cosine learning-rate schedule and **no EMA** (exponential moving average of weights). Sources: [DDPM](https://arxiv.org/abs/2006.11239) sections 2 and 3, equations 4 and 12, Algorithm 1; [Diffusion Policy](https://diffusion-policy.cs.columbia.edu/).

**Supplied here:** the beta schedule, the denoiser network (`ConditionalUnet1D`, which you write in Unit 10), the data reader and the training loop (`optimizer_update`, `train_loop` from the training unit). **You write** the five functions that turn a clean window into a training signal.

**Real data used in this unit.** Real LIBERO `bowl_plate` demonstrations are read from local files (no download starts here). A *window* has the robot state at the last 2 frames and 16 actions that start one frame before the current one. The first window slot is an action *already taken*; this is the pinned alignment, and Unit 11 uses it to pick the executed actions. Windows near an episode edge repeat the nearest frame and set `action_is_pad=True`. The recording rate is 10 Hz; nothing is resampled to the simulator's 20 Hz controller. Values are scaled to `[-1, 1]` with training-set extrema. If the files are missing, the cell tells you the setup command.

In [ ]:
# Supplied: real windows (train and validation episodes are kept apart)
TRAIN = load_state_action_windows(CONFIGS, DATA, "bowl_plate", "train")
VALID = load_state_action_windows(CONFIGS, DATA, "bowl_plate", "validation")
print("train windows:", len(TRAIN["action"]), "| validation windows:", len(VALID["action"]))
print("state:", tuple(VALID["observation.state"].shape[1:]), "| action:", tuple(VALID["action"].shape[1:]))
EXAMPLE = int((VALID["action_is_pad"].sum(1) == 4).nonzero()[0])   # a window whose last 4 steps are padding
example = VALID["action"][EXAMPLE]

In [ ]:
# Supplied: a small real pinned DiffusionPolicy and a batch for algorithm checks
IMAGES = ("observation.images.image", "observation.images.image2")


def tiny_policy(seed=5):
    """A small *real* pinned DiffusionPolicy with random weights (64x64 images, horizon 16)."""
    torch.manual_seed(seed)
    config = DiffusionConfig(
        input_features={
            IMAGES[0]: PolicyFeature(FeatureType.VISUAL, (3, 64, 64)),
            IMAGES[1]: PolicyFeature(FeatureType.VISUAL, (3, 64, 64)),
            "observation.state": PolicyFeature(FeatureType.STATE, (8,)),
        },
        output_features={"action": PolicyFeature(FeatureType.ACTION, (7,))},
        n_obs_steps=2, horizon=16, n_action_steps=8, drop_n_last_frames=7,
        down_dims=(16, 32, 64), spatial_softmax_num_keypoints=8, diffusion_step_embed_dim=32,
        pretrained_backbone_weights=None, crop_is_random=False, num_train_timesteps=100,
        num_inference_steps=20, do_mask_loss_for_padding=True, push_to_hub=False, device="cpu",
    )
    policy = DiffusionPolicy(config)
    for parameter in policy.parameters():
        if parameter.dim() == 1:
            nn.init.normal_(parameter, std=0.05)
    return policy.eval()


def algorithm_check_batch(batch=3, seed=1):
    """Random numbers in the right shapes. Clearly labelled: an algorithm check, not data."""
    g = torch.Generator().manual_seed(seed)
    pad = torch.zeros(batch, 16, dtype=torch.bool)
    pad[1, 12:] = True
    pad[2, 3:] = True
    return {
        "observation.state": torch.randn(batch, 2, 8, generator=g),
        IMAGES[0]: torch.rand(batch, 2, 3, 64, 64, generator=g),
        IMAGES[1]: torch.rand(batch, 2, 3, 64, 64, generator=g),
        "action": torch.rand(batch, 16, 7, generator=g) * 2 - 1,
        "action_is_pad": pad,
    }


POLICY = tiny_policy()
CHECK_BATCH = algorithm_check_batch()

## Exercise 9.1 — Cumulative alphas (about 10 min)

**Purpose.** The noise level at step `t` depends on the product of all `(1 − β)` up to `t`. Compute it once for all steps.

**Contract (shapes, units, masks).**

- `cumulative_alphas(betas)`: `betas` is a `float32` tensor `(T,)` with values in `(0, 1)`.
- Return `alpha_bar` `(T,)`, `float32`, with `alpha_bar[t] = prod_{s <= t} (1 - betas[s])` (includes step `t`). So `alpha_bar[0] = 1 - betas[0]`.
- Do not loop in Python over steps; one tensor operation is enough.

**Method and source.** `α_t = 1 − β_t`, `ᾱ_t = ∏_{s=1}^{t} α_s` — [DDPM](https://arxiv.org/abs/2006.11239), section 2, just before equation 4.

**Pointers.** Turn betas into alphas first. Then use a cumulative product along the only axis.

<details><summary>Detailed hint</summary>

`torch.cumprod(1.0 - betas, dim=0)`.

</details>

*This is the reference notebook: the next cell is a complete solution.*

In [ ]:
# Reference solution for Exercise 9.1
from __future__ import annotations

def cumulative_alphas(betas: Tensor) -> Tensor:
    """``alpha_bar[t] = prod_{s <= t} (1 - beta[s])`` (DDPM paper, equation 4). Shape ``(T,)``."""
    return torch.cumprod(1.0 - betas, dim=0)

In [ ]:
# Check: Exercise 9.1
betas = cosine_betas(100)
alpha_bar = cumulative_alphas(betas)
assert alpha_bar.shape == (100,) and alpha_bar.dtype == torch.float32, "Expected a float32 tensor of shape (T,)."
close(alpha_bar[0], 1 - betas[0], "alpha_bar[0] is 1 - beta[0]")
close(alpha_bar[2], (1 - betas[0]) * (1 - betas[1]) * (1 - betas[2]), "alpha_bar[2] is the product of three alphas")
assert (alpha_bar[1:] < alpha_bar[:-1]).all(), "alpha_bar must decrease with t."
scheduler = DDPMScheduler(num_train_timesteps=100, beta_schedule="squaredcos_cap_v2")
close(alpha_bar, scheduler.alphas_cumprod, "differs from diffusers' alphas_cumprod", atol=1e-7)
close(cumulative_alphas(torch.tensor([0.5, 0.5, 0.5])), torch.tensor([0.5, 0.25, 0.125]), "constant betas", atol=1e-7)
print("Exercise 9.1 passed")

**Use your function.** Compare two noise schedules: the cosine schedule used by the pinned policy and a linear one. Where is the signal gone?

In [ ]:
# Use your function: Exercise 9.1
linear_betas = torch.linspace(1e-4, 0.02, 100)
for name, b in (("cosine (pinned)", cosine_betas(100)), ("linear 1e-4..0.02", linear_betas)):
    plt.plot(cumulative_alphas(b), label=name)
plt.xlabel("timestep t"); plt.ylabel("alpha_bar[t] (signal kept)"); plt.legend(); plt.grid(alpha=0.3); plt.show()
print("alpha_bar at the last step: cosine", float(cumulative_alphas(cosine_betas(100))[-1]), "| linear", float(cumulative_alphas(linear_betas)[-1]))

## Exercise 9.2 — Forward noising (about 15 min)

**Purpose.** Jump from a clean action window to its noisy version at any step in one operation.

**Contract (shapes, units, masks).**

- `x0` and `noise`: `(B, H, A)`. `timesteps`: `(B,)` `long`, values in `[0, T)`. `alpha_bar`: `(T,)`.
- Each batch row uses its *own* timestep.
- Return `x_t` `(B, H, A)` with `x_t = sqrt(alpha_bar[t]) * x0 + sqrt(1 - alpha_bar[t]) * noise`.
- Do not change `x0` or `noise` in place. Gradients must flow to `x0` and `noise`.

**Method and source.** `q(x_t | x_0) = N(√ᾱ_t x_0, (1 − ᾱ_t) I)` — [DDPM](https://arxiv.org/abs/2006.11239), equation 4. This is also what `DDPMScheduler.add_noise` computes.

**Pointers.** Look up the per-row coefficients with the timestep tensor. They have shape `(B,)`; reshape them so they broadcast against `(B, H, A)`.

<details><summary>Detailed hint</summary>

```python
abar = alpha_bar.to(device=x0.device, dtype=x0.dtype)[timesteps]       # (B,)
abar = abar.reshape(-1, 1, 1)                                          # (B, 1, 1)
return abar.sqrt() * x0 + (1.0 - abar).sqrt() * noise
```

</details>

*This is the reference notebook: the next cell is a complete solution.*

In [ ]:
# Reference solution for Exercise 9.2
from __future__ import annotations

def forward_noise(x0: Tensor, noise: Tensor, timesteps: Tensor, alpha_bar: Tensor) -> Tensor:
    """Noise a clean sample in one step (DDPM equation 4).

    ``x_t = sqrt(alpha_bar[t]) * x0 + sqrt(1 - alpha_bar[t]) * noise``, with ``t`` chosen per
    batch row. ``x0`` and ``noise`` are ``(B, H, A)``, ``timesteps`` is ``(B,)``.
    """
    abar = alpha_bar.to(device=x0.device, dtype=x0.dtype)[timesteps].reshape(
        -1, *([1] * (x0.ndim - 1))
    )
    return abar.sqrt() * x0 + (1.0 - abar).sqrt() * noise

In [ ]:
# Check: Exercise 9.2
scheduler = DDPMScheduler(num_train_timesteps=100, beta_schedule="squaredcos_cap_v2")
alpha_bar = cumulative_alphas(cosine_betas(100))
x0, noise = torch.randn(4, 16, 7, requires_grad=True), torch.randn(4, 16, 7)
t = torch.tensor([0, 17, 50, 99])
out = forward_noise(x0, noise, t, alpha_bar)
assert out.shape == x0.shape, "Return the same shape as x0."
close(out, scheduler.add_noise(x0.detach(), noise, t), "x_t differs from diffusers' add_noise")
row = forward_noise(x0[1:2].detach(), noise[1:2], torch.tensor([17]), alpha_bar)
close(out[1:2], row, "a row must depend only on its own timestep")
low = forward_noise(x0.detach(), noise, torch.zeros(4, dtype=torch.long), alpha_bar)
assert (low - x0.detach()).abs().max() < 0.25, "At t = 0 the sample is almost the clean window."
out.sum().backward()
close(x0.grad[0], alpha_bar[0].sqrt().expand(16, 7), "d x_t / d x0 is sqrt(alpha_bar[t])", atol=1e-6)
print("Exercise 9.2 passed")

**Use your function.** Noise a real action window at five steps with the same fixed noise, and look at one action channel.

In [ ]:
# Use your function: Exercise 9.2
alpha_bar = cumulative_alphas(cosine_betas(100))
noise = torch.randn(1, 16, 7, generator=torch.Generator().manual_seed(0))
steps = [0, 25, 50, 75, 99]
fig, axis = plt.subplots(figsize=(7, 3.2))
for t in steps:
    noisy = forward_noise(example[None], noise, torch.tensor([t]), alpha_bar)[0]
    axis.plot(noisy[:, 0], label=f"t={t}")
axis.plot(example[:, 0], "k--", linewidth=2, label="clean")
axis.axvspan(12 - 0.5, 15.5, color="gray", alpha=0.15, label="padded steps")
axis.set_xlabel("action step"); axis.set_ylabel("channel 0"); axis.legend(ncol=3, fontsize=8); plt.show()

## Exercise 9.3 — Choose noise and timesteps (about 10–15 min)

**Purpose.** Each training example needs its own noise and its own random noise level. Draw both in a fixed order so a run can be repeated exactly.

**Contract (shapes, units, masks).**

- `actions (B, H, A)` float tensor, `num_train_timesteps` (int `T`), `generator` (a `torch.Generator` on the same device as `actions`, or `None` for the global generator).
- Draw **first** `noise ~ N(0, I)` with the shape, dtype and device of `actions`; **then** `timesteps`, `B` integers drawn uniformly from `0 … T-1` (`T` itself is excluded), dtype `long`, on the same device.
- Use the same `generator` for both draws. Return `(noise, timesteps)`.

**Method and source.** Algorithm 1 in [DDPM](https://arxiv.org/abs/2006.11239): `t ~ Uniform({1, …, T})`, `ε ~ N(0, I)`. Indices here start at 0. The pinned `compute_loss` draws noise first, then `torch.randint`.

**Pointers.** Two calls: `torch.randn` with an explicit shape, `torch.randint` with `low`, `high` and a size. Pass the generator to both.

<details><summary>Detailed hint</summary>

```python
noise = torch.randn(actions.shape, generator=generator, device=actions.device, dtype=actions.dtype)
timesteps = torch.randint(0, num_train_timesteps, (actions.shape[0],), generator=generator, device=actions.device)
return noise, timesteps
```

</details>

*This is the reference notebook: the next cell is a complete solution.*

In [ ]:
# Reference solution for Exercise 9.3
from __future__ import annotations

def sample_noise_and_timesteps(
    actions: Tensor, num_train_timesteps: int, generator: torch.Generator | None = None
) -> tuple[Tensor, Tensor]:
    """Draw ``noise ~ N(0, I)`` shaped like ``actions``, then one uniform timestep per row.

    Timesteps are integers in ``[0, T)`` (``T`` itself is excluded), dtype ``long``, on the
    device of ``actions``. Draw the noise first, then the timesteps, from the same generator.
    """
    noise = torch.randn(
        actions.shape, generator=generator, device=actions.device, dtype=actions.dtype
    )
    timesteps = torch.randint(
        0, num_train_timesteps, (actions.shape[0],), generator=generator, device=actions.device
    )
    return noise, timesteps

In [ ]:
# Check: Exercise 9.3
actions = torch.zeros(2000, 16, 7)
noise, t = sample_noise_and_timesteps(actions, 100, torch.Generator().manual_seed(3))
assert noise.shape == actions.shape and t.shape == (2000,) and t.dtype == torch.long, "Expected noise like actions and long timesteps (B,)."
assert t.min() == 0 and t.max() == 99, "Timesteps must cover 0 … T-1 and never reach T."
assert abs(float(noise.mean())) < 0.01 and abs(float(noise.std()) - 1) < 0.01, "Noise must be standard normal."
again = sample_noise_and_timesteps(actions, 100, torch.Generator().manual_seed(3))
assert torch.equal(noise, again[0]) and torch.equal(t, again[1]), "The same seeded generator must give the same draws."
torch.manual_seed(9); expected_noise = torch.randn(actions.shape); expected_t = torch.randint(0, 100, (2000,))
torch.manual_seed(9); drawn = sample_noise_and_timesteps(actions, 100)
assert torch.equal(drawn[0], expected_noise) and torch.equal(drawn[1], expected_t), "Draw the noise first, then the timesteps (the pinned order)."
print("Exercise 9.3 passed")

**Use your function.** Draw the noise levels for a real training batch and check that all levels are used about equally.

In [ ]:
# Use your function: Exercise 9.3
batch = take_batch(TRAIN, torch.arange(512))
noise, timesteps = sample_noise_and_timesteps(batch["action"], 100, torch.Generator().manual_seed(0))
plt.hist(timesteps.numpy(), bins=20); plt.xlabel("timestep"); plt.ylabel("count in a batch of 512"); plt.show()
print("noise shape:", tuple(noise.shape), "| first timesteps:", timesteps[:8].tolist())

## Exercise 9.4 — Masked noise loss (about 10–15 min)

**Purpose.** Padded action steps are copies, not decisions. They must not train the network. This is the loss that ignores them.

**Contract (shapes, units, masks).**

- `prediction`, `target`: `(B, H, A)`. `is_pad`: `(B, H)` bool, `True` = padded step.
- Return one scalar tensor: `sum((prediction - target)**2 * valid) / max(valid_steps * A, 1)`, where `valid = ~is_pad` is broadcast over `A` and `valid_steps` counts the valid `(row, step)` pairs in the whole batch.
- Changing `prediction` or `target` only at padded steps must change neither the loss nor the gradient. A batch that is entirely padding returns `0` (no `nan`).

**Method and source.** Mean squared error `‖ε − ε_θ‖²` ([DDPM](https://arxiv.org/abs/2006.11239), equation 12 / Algorithm 1) averaged over real elements only; the pinned `compute_loss` does `(loss * mask).sum() / (mask.sum() * action_dim).clamp_min(1)`.

**Pointers.** Square the error. Multiply by a float mask. Divide by the number of valid *scalar* values, not the number of rows.

<details><summary>Detailed hint</summary>

```python
valid = (~is_pad.to(torch.bool)).unsqueeze(-1).to(prediction.dtype)        # (B, H, 1)
denominator = (valid.sum() * prediction.shape[-1]).clamp_min(1)
return ((prediction - target).square() * valid).sum() / denominator
```

</details>

*This is the reference notebook: the next cell is a complete solution.*

In [ ]:
# Reference solution for Exercise 9.4
from __future__ import annotations

def masked_epsilon_loss(prediction: Tensor, target: Tensor, is_pad: Tensor) -> Tensor:
    """Mean squared error over valid ``(B, H, A)`` elements.

    ``sum((prediction - target) ** 2 * valid) / max(number_of_valid_steps * A, 1)`` with
    ``valid = ~is_pad`` broadcast over ``A``. Padded steps have zero gradient. A batch that is
    entirely padding returns zero.
    """
    valid = (~is_pad.to(torch.bool)).unsqueeze(-1).to(prediction.dtype)
    denominator = (valid.sum() * prediction.shape[-1]).clamp_min(1)
    return ((prediction - target).square() * valid).sum() / denominator

In [ ]:
# Check: Exercise 9.4
pred = torch.zeros(2, 3, 2, requires_grad=True)
target = torch.ones(2, 3, 2)
pad = torch.tensor([[False, False, True], [False, True, True]])
loss = masked_epsilon_loss(pred, target, pad)
assert loss.shape == () and torch.isfinite(loss), "Return one finite scalar."
close(loss, torch.tensor(1.0), "three valid steps, each with error 1")
loss.backward()
assert torch.count_nonzero(pred.grad[pad]) == 0, "Padded steps must have zero gradient."
close(pred.grad[~pad].sum(), torch.tensor(-2 * 6 / 6.0), "gradient of the valid elements", atol=1e-6)
shaken = pred.detach().clone(); shaken[pad] = 99.0
close(masked_epsilon_loss(shaken, target, pad), loss.detach(), "changing padded predictions must not change the loss")
assert masked_epsilon_loss(pred.detach(), target, torch.ones_like(pad)) == 0, "An all-padding batch gives exactly 0."
print("Exercise 9.4 passed")

**Use your function.** On real windows that contain padding, build a prediction that is perfect on real steps and wrong only on padded steps. A correct masked loss is exactly 0; a plain mean is not.

In [ ]:
# Use your function: Exercise 9.4
has_pad = VALID["action_is_pad"].any(1)
batch = take_batch(VALID, has_pad.nonzero().flatten())
noise = torch.randn(batch["action"].shape, generator=torch.Generator().manual_seed(1))
prediction = noise.clone()
prediction[batch["action_is_pad"]] = 0.0              # wrong only where the step is padded
print("windows with padding:", len(noise), "| padded fraction of their steps:", round(float(batch["action_is_pad"].float().mean()), 3))
print("masked loss:", float(masked_epsilon_loss(prediction, noise, batch["action_is_pad"])))
print("plain mean :", float((prediction - noise).square().mean()))

## Exercise 9.5 — The loss callback and one denoiser update (about 20 min)

**Purpose.** Combine your four functions into the objective that the training loop calls. The loop owns the optimizer step; you own the loss.

**Contract (shapes, units, masks).**

- `loss_fn(model, batch, *, noise=None, time=None) -> (scalar_loss, metrics)`. The loss tensor keeps its autograd graph; `metrics` is a dict of scalar tensors.
- `model(noisy_actions, timesteps, batch)` predicts the noise and `model.schedule` holds `.alpha_bar (T,)` and `.num_train_timesteps`.
- `batch["action"]` is `(B, H, A)`; `batch["action_is_pad"]` is `(B, H)`. The model reads the observations from `batch` itself.
- If `noise` and/or `time` (the timesteps) are given, use them exactly. Draw whichever is missing with `sample_noise_and_timesteps`, **noise first**.
- Steps: `noisy = forward_noise(actions, noise, time, alpha_bar)`; `prediction = model(noisy, time, batch)`; `loss = masked_epsilon_loss(prediction, noise, batch["action_is_pad"])`. Return `loss, {"epsilon_mse": loss}`.

**Method and source.** [DDPM](https://arxiv.org/abs/2006.11239) Algorithm 1: sample, noise, predict, regress on the noise. The training unit's `optimizer_update(model, batch, optimizer, loss_fn, noise=…, time=…)` calls `loss_fn`, runs `backward()` and steps the optimizer.

**Pointers.** Draw only what is missing. Do not detach the loss. The padded mask is `batch['action_is_pad']`.

<details><summary>Detailed hint</summary>

```python
actions = batch["action"]
drawn_noise, drawn_time = sample_noise_and_timesteps(actions, model.schedule.num_train_timesteps)
noise = drawn_noise if noise is None else noise
time = drawn_time if time is None else time
noisy = forward_noise(actions, noise, time, model.schedule.alpha_bar)
loss = masked_epsilon_loss(model(noisy, time, batch), noise, batch["action_is_pad"])
return loss, {"epsilon_mse": loss}
```
Drawing both and using only what is missing keeps the random stream the same as the pinned code.

</details>

*This is the reference notebook: the next cell is a complete solution.*

In [ ]:
# Reference solution for Exercise 9.5
from __future__ import annotations

def epsilon_loss_fn(
    model: nn.Module,
    batch: Mapping[str, Tensor],
    *,
    noise: Tensor | None = None,
    time: Tensor | None = None,
) -> tuple[Tensor, dict[str, Tensor]]:
    """Training objective callback: ``(loss, metrics)`` with the graph kept for ``backward``.

    ``model(noisy_actions, timesteps, batch)`` predicts the noise and ``model.schedule`` holds
    ``alpha_bar``. Use ``noise`` and ``time`` (timesteps) when given; draw whichever is missing.
    """
    actions = batch["action"]
    if noise is None:
        noise = torch.randn(actions.shape, device=actions.device, dtype=actions.dtype)
    if time is None:
        time = torch.randint(
            0,
            model.schedule.num_train_timesteps,
            (actions.shape[0],),
            device=actions.device,
        )
    noisy = forward_noise(actions, noise, time, model.schedule.alpha_bar)
    loss = masked_epsilon_loss(model(noisy, time, batch), noise, batch["action_is_pad"])
    return loss, {"epsilon_mse": loss}

In [ ]:
# Check: Exercise 9.5
denoiser = LeRobotDenoiser(POLICY)            # the pinned network, wrapped so that it predicts noise
batch = CHECK_BATCH
torch.manual_seed(21)
expected = POLICY.diffusion.compute_loss(stack_observation(POLICY, batch))     # the pinned loss
torch.manual_seed(21)
loss, metrics = epsilon_loss_fn(denoiser, batch)
assert loss.shape == () and "epsilon_mse" in metrics, "Return (scalar loss, metrics with epsilon_mse)."
close(loss, expected, "your loss differs from the pinned compute_loss for the same random numbers", atol=1e-6)
noise, time = torch.randn_like(batch["action"]), torch.tensor([3, 40, 90])
a, _ = epsilon_loss_fn(denoiser, batch, noise=noise, time=time)
b, _ = epsilon_loss_fn(denoiser, batch, noise=noise, time=time)
close(a, b, "explicit noise and time must make the loss repeatable", atol=1e-7)
by_hand = masked_epsilon_loss(denoiser(forward_noise(batch["action"], noise, time, denoiser.schedule.alpha_bar), time, batch),
                              noise, batch["action_is_pad"])
close(a, by_hand, "the loss with explicit noise and time", atol=1e-6)
a.backward()
assert any(p.grad is not None and p.grad.abs().sum() > 0 for p in POLICY.diffusion.unet.parameters()), "The loss must keep its graph (U-Net gradients)."
assert any(p.grad is not None and p.grad.abs().sum() > 0 for p in POLICY.diffusion.rgb_encoder.parameters()), "Gradients must also reach the image encoders."
POLICY.zero_grad()
print("Exercise 9.5 passed")

**Use your function.** Now use the loss to train a denoiser with the training unit's loop: a small U-Net conditioned on the real robot state, on real windows. Watch the loss, then ask which noise levels are hard.

In [ ]:
# Use your function: Exercise 9.5
torch.manual_seed(0)
schedule = DDPMSchedule.cosine(100)
unet = ConditionalUnet1D(7, 2 * 8, (32, 64, 128), step_embed_dim=32)       # supplied denoiser; you write it in Unit 10
model = DiffusionDenoiser(unet, schedule, flat_state_condition)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, betas=(0.95, 0.999))
draws = torch.Generator().manual_seed(1)
batches = [take_batch(TRAIN, torch.randint(0, len(TRAIN["action"]), (64,), generator=draws)) for _ in range(3)]
history = train_loop(model, batches, optimizer, epsilon_loss_fn, steps=3)
plt.plot([h["loss"] for h in history]); plt.xlabel("update"); plt.ylabel("masked epsilon MSE (three-update integration check)"); plt.show()
model.eval()
with torch.no_grad():
    fixed_noise = torch.randn(VALID["action"].shape, generator=torch.Generator().manual_seed(7))
    per_step = [float(epsilon_loss_fn(model, VALID, noise=fixed_noise, time=torch.full((len(fixed_noise),), t))[0]) for t in range(0, 100, 11)]
plt.plot(range(0, 100, 11), per_step, marker="o"); plt.xlabel("timestep"); plt.ylabel("validation epsilon MSE"); plt.show()

## Experiment — which noise schedule is easier to learn? (30–60 min)

**Question.** Under the same update budget, how do the cosine schedule and two linear schedules compare at *matched noise levels*, and does masking the padded steps matter?

**Why "matched noise levels".** Timestep `t = 50` means different amounts of noise under different schedules, so comparing the loss at the same `t` is unfair. Compare at the same `ᾱ` (signal kept): for each schedule pick the step whose `ᾱ` is closest to `0.9` and to `0.5`. Also read `final_alpha_bar`: a schedule whose last step still keeps much signal ends before pure noise, which matters for sampling in Unit 11.

**Setup (real data).** The windows above, a small U-Net conditioned on the flattened 2×8 state history (`ConditionalUnet1D`, supplied), your functions `cumulative_alphas`, `forward_noise`, `sample_noise_and_timesteps`, `masked_epsilon_loss` and `epsilon_loss_fn` inside the real update (`train_loop`).

**Fixed for every trial.** Weight seed `0`; batch seed `1`; **200 updates**, batch 64, Adam(`lr=1e-3`, betas `(0.95, 0.999)`); `T = 100`; validation noise seed `7` and the same validation windows for every trial.

**Allowed changes (per trial).** `schedule`: `cosine`, `linear` (β from `1e-4` to `0.02`) or `linear_strong` (β from `1e-4` to `0.1`). `mask`: `True` uses the real padding mask; `False` pretends no step is padded (the padded copies then train the network).

**Measures** (validation windows, real steps only, fixed noise): `eps@0.9`, `eps@0.5` — noise-prediction MSE; `x0@0.9`, `x0@0.5` — MSE of the clean-window estimate `(x_t − √(1−ᾱ) ε̂) / √ᾱ`; `final_alpha_bar`.

**Prediction first.** Edit the next cell: which schedule will have the lowest `eps@0.5`, and will `mask=False` change the measures much?

**Keep failed trials.** Every trial is appended to `TRIALS`. No improvement is a valid result.

In [ ]:
# Larger training experiment: run only when you opt in.
RUN_EXPERIMENT = False

In [ ]:
if RUN_EXPERIMENT:
    PREDICTION = "TODO: replace this text with your prediction."
    print(PREDICTION)

In [ ]:
if RUN_EXPERIMENT:
    # Supplied: trial table helper
    import time
    import pandas as pd

    def show_trials(trials, columns):
        """Print every trial that was run, including failed ones, and return a table."""
        table = pd.DataFrame([{**t["config"], **{c: t.get(c) for c in columns}, "status": t["status"]} for t in trials])
        print(table.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
        return table

In [ ]:
if RUN_EXPERIMENT:
    # Supplied: harness. Your functions run inside every update.
    STEPS, BATCH, LR, SEED_WEIGHTS, SEED_BATCH, SEED_NOISE = 200, 64, 1e-3, 0, 1, 7


    def make_schedule(name):
        if name == "cosine":
            return DDPMSchedule.from_betas(cosine_betas(100), cumulative_fn=cumulative_alphas)
        end = {"linear": 0.02, "linear_strong": 0.1}[name]
        return DDPMSchedule.from_betas(torch.linspace(1e-4, end, 100), cumulative_fn=cumulative_alphas)


    def matched_step(schedule, level):
        return int((schedule.alpha_bar - level).abs().argmin())


    BASELINE = {"schedule": "cosine", "mask": True}
    TRIALS = []


    def run_trial(config):
        started = time.perf_counter()
        torch.manual_seed(SEED_WEIGHTS)
        schedule = make_schedule(config["schedule"])
        model = DiffusionDenoiser(ConditionalUnet1D(7, 16, (32, 64, 128), step_embed_dim=32), schedule, flat_state_condition)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.95, 0.999))
        draws = torch.Generator().manual_seed(SEED_BATCH)
        batches = []
        for _ in range(STEPS):
            batch = take_batch(TRAIN, torch.randint(0, len(TRAIN["action"]), (BATCH,), generator=draws))
            if not config["mask"]:
                batch["action_is_pad"] = torch.zeros_like(batch["action_is_pad"])
            batches.append(batch)
        train_loop(model, batches, optimizer, epsilon_loss_fn, steps=STEPS)
        model.eval()
        result = {}
        with torch.no_grad():
            noise = torch.randn(VALID["action"].shape, generator=torch.Generator().manual_seed(SEED_NOISE))
            real = ~VALID["action_is_pad"]
            for level in (0.9, 0.5):
                t = matched_step(schedule, level)
                times = torch.full((len(noise),), t)
                noisy = forward_noise(VALID["action"], noise, times, schedule.alpha_bar)
                eps = model(noisy, times, VALID)
                abar = schedule.alpha_bar[t]
                x0 = (noisy - (1 - abar).sqrt() * eps) / abar.sqrt()
                result[f"eps@{level}"] = (eps - noise).square().mean(-1)[real].mean().item()
                result[f"x0@{level}"] = (x0 - VALID["action"]).square().mean(-1)[real].mean().item()
        result["final_alpha_bar"] = float(schedule.alpha_bar[-1])
        result["seconds"] = time.perf_counter() - started
        return result


    def record(config):
        config = {**BASELINE, **config}
        try:
            TRIALS.append({"config": config, "status": "ok", **run_trial(config)})
        except Exception as error:  # a failed trial is evidence too
            TRIALS.append({"config": config, "status": f"failed: {type(error).__name__}: {error}"})


    record(BASELINE)
    COLUMNS = ["eps@0.9", "eps@0.5", "x0@0.9", "x0@0.5", "final_alpha_bar", "seconds"]
    show_trials(TRIALS, COLUMNS);

**Your trials.** Add at least two trials that each change one setting. Budget: 200 updates per trial.

In [ ]:
if RUN_EXPERIMENT:
    # Reference choice of trials: one change per trial.
    MY_TRIALS = [{"schedule": "linear"}, {"schedule": "linear_strong"}, {"mask": False}]
    for trial in MY_TRIALS:
        record(trial)
    show_trials(TRIALS, COLUMNS);

In [ ]:
if RUN_EXPERIMENT:
    # Check: the protocol was followed.
    assert TRIALS[0]["config"] == BASELINE, "The first trial must be the baseline."
    assert len(TRIALS) >= 3, "Run the baseline and at least two trials of your own."
    assert all(set(t["config"]) == set(BASELINE) for t in TRIALS), "Only schedule and mask may change."
    assert all(t["config"]["schedule"] in {"cosine", "linear", "linear_strong"} for t in TRIALS)
    ok = [t for t in TRIALS if t["status"] == "ok"]
    assert all(torch.isfinite(torch.tensor([t[c] for c in COLUMNS])).all() for t in ok), "Measures must be finite."
    print(f"{len(TRIALS)} trials kept ({len(ok)} ok). Budget: {STEPS} updates, batch {BATCH}, seeds {SEED_WEIGHTS}/{SEED_BATCH}/{SEED_NOISE}.")

In [ ]:
if RUN_EXPERIMENT:
    # Use your function: plot every trial
    ok = [t for t in TRIALS if t["status"] == "ok"]
    labels = [", ".join(f"{k}={v}" for k, v in t["config"].items() if v != BASELINE[k]) or "baseline" for t in ok]
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
    for axis, column in zip(axes, ["eps@0.5", "x0@0.5", "final_alpha_bar"]):
        axis.bar(range(len(ok)), [t[column] for t in ok])
        axis.set_xticks(range(len(ok))); axis.set_xticklabels(labels, rotation=30, ha="right"); axis.set_title(column)
    plt.tight_layout(); plt.show()

**Explain (2–4 sentences, edit this cell).** Which schedule had the lowest loss at matched noise, and is the comparison fair for the schedule that ends with `final_alpha_bar` far from 0? Did masking change the validation numbers, which count real steps only? What is one limit of using the state alone (no images) as the condition?

*Optional extension:* run the cosine baseline with two more batch seeds and report the spread of `eps@0.5`; compare it with the gaps between schedules.

## Optional: your loss on the real prepared policy (opt-in)

This loads the **prepared** diffusion checkpoint `diffusion-bowl-plate-reference-2000` on CPU, reads two real training windows with their camera frames, and runs your `epsilon_loss_fn` through the pinned network (image encoders + U-Net) for the same random numbers as the pinned loss. It then makes **two Adam updates** with your loss through the training loop, so you see that your callback updates the real weights. Two updates are an integration check, not training evidence; nothing is saved. The run uses local files and two training windows. Set `RUN_REAL_DIFFUSION_UPDATE = True` to run it.

In [ ]:
RUN_REAL_DIFFUSION_UPDATE = False

if RUN_REAL_DIFFUSION_UPDATE:
    import copy
    from torch.utils.data import default_collate
    from vla_lab.teaching.act_model import open_libero_dataset
    from vla_lab.teaching.diffusion import load_prepared_diffusion
    from vla_lab.training.diffusion import FPS, diffusion_delta_timestamps

    checkpoint = DATA / "runs/diffusion-bowl-plate-reference-2000/checkpoint-002000"
    if not (checkpoint / "model.safetensors").is_file():
        raise FileNotFoundError(f"Prepared diffusion checkpoint missing: {checkpoint}")
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    policy, preprocessor, _ = load_prepared_diffusion(checkpoint, device=DEVICE)
    dataset = open_libero_dataset(CONFIGS, DATA, "bowl_plate", "train", diffusion_delta_timestamps(policy.config, FPS), episodes=[379, 422])
    batch = preprocessor(default_collate([dataset[i] for i in (3, 40)]))
    denoiser = LeRobotDenoiser(policy)
    policy.eval()
    torch.manual_seed(2); pinned_loss, _ = policy(dict(batch))
    torch.manual_seed(2); your_loss, _ = epsilon_loss_fn(denoiser, batch)
    print("pinned loss:", float(pinned_loss.detach()), "| your loss:", float(your_loss.detach()))
    model = LeRobotDenoiser(policy)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-5, betas=(0.95, 0.999))
    before = policy.diffusion.unet.final_conv[1].weight.detach().clone()
    history = train_loop(model, [batch], optimizer, epsilon_loss_fn, steps=2)
    print("two updates, losses:", [round(h["loss"], 4) for h in history])
    print("U-Net weights changed:", not torch.equal(before, policy.diffusion.unet.final_conv[1].weight))
else:
    print("Real diffusion update is off. Set RUN_REAL_DIFFUSION_UPDATE=True to run the two-update integration check.")

## What you built

- `cumulative_alphas`, `forward_noise`, `sample_noise_and_timesteps`, `masked_epsilon_loss` and `epsilon_loss_fn`: the full training objective, equal to the pinned one for equal random numbers.
- A fair comparison across noise schedules (matched `ᾱ`) with every trial kept.

**Next:** Unit 10A opens the denoiser: the timestep embedding and the FiLM-conditioned residual block.